# Krok 2. Średnia czy mediana (E1, E4)
**Wynik z instrukcji:** wybór „średnia czy mediana” z uzasadnieniem (udział zer, skośność). Poziom: typ SKU. Dla wszystkich wariantów.

**Pytanie.** Gdy prognozą jest „typowa sprzedaż towaru”, można ją liczyć jako średnią albo medianę. Mediana jest odporna na pojedyncze skoki, ale gdy towar sprzedaje się rzadko, większość tygodni to zera i mediana wynosi 0, a prognoza zero jest bezużyteczna. Średnia jest wrażliwa na skoki, ale zachowuje sumę sprzedaży. Wybór zależy od typu popytu, więc sprawdzamy go osobno dla każdego typu SKU.

**Test (tylko lata EDA).** Uczymy się na 2023 (średnia i mediana tygodni towaru), a sprawdzamy na 2024. Typ popytu wyznaczamy **tylko z 2023**, żeby podział nie podpatrywał 2024.

**Wynik zapisany do:** `dane/eda/wynik_02_srednia_mediana.csv`.

In [9]:
# Ustawienia wspólne: moduł eda_wspolne.py leży obok tego notebooka
import os, sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import eda_wspolne as ew

ew.wejdz_do_korzenia()      # przechodzi do katalogu głównego projektu (tam, gdzie jest plik z danymi)
ew.styl()
print("Katalog projektu:", os.getcwd(), "| EDA używa danych do", ew.KONIEC_EDA.date(), "(rok testowy", ew.ROK_TEST, "pomijamy)")

Katalog projektu: /Users/mixer/python/analiza_sprzedazy_w_sklepie | EDA używa danych do 2024-12-31 (rok testowy 2025 pomijamy)


## Dane i typ popytu
#### Objaśnienie
**Po co.** Potrzebujemy sprzedaży tygodniowej każdego towaru i jego typu popytu.
**Typy popytu (Syntetos-Boylan).** Dwie liczby opisują towar: **ADI** (średni odstęp w tygodniach między tygodniami ze sprzedażą) i **CV²** (jak bardzo różnią się wielkości sprzedaży, gdy ona jest). Progi 1,32 i 0,49 dają cztery typy: *regularny* (często i podobnie), *zmienny* (często, ale różne wielkości), *sporadyczny* (rzadko, podobne wielkości), *nieregularny* (rzadko i różne wielkości). Towary z mniej niż 13 tygodniami od pierwszej sprzedaży dostają `za_malo_danych`.
**Jak czytać wynik.** Tabela pokazuje, ile jest towarów każdego typu i jaki mają udział w sprzedaży. Typów rzadkich będzie dużo pod względem liczby, a mało pod względem wolumenu.

In [10]:
fa = ew.wczytaj_sprzedaz()
kal = ew.kalendarz()
tyg = ew.panel_tygodniowy(fa, kal, "SKU")             # towary x tygodnie
del fa
T = 52                                                 # uczymy na pierwszych 52 tygodniach (2023), testujemy na kolejnych 52 (2024)
assert tyg.shape[1] == 104, f"oczekiwano 104 tygodni, jest {tyg.shape[1]} ({tyg.columns[0].date()} … {tyg.columns[-1].date()})"
akt = ew.aktywne_tygodnie(tyg)
typ = ew.typ_popytu(tyg.iloc[:, :T])                   # typ z 2023: nie widzi 2024
tab_typ = typ.groupby("typ").agg(towary=("sprzedaz", "size"), sprzedaz=("sprzedaz", "sum"))
tab_typ["udzial_sprzedazy_%"] = 100 * tab_typ["sprzedaz"] / tab_typ["sprzedaz"].sum()
print(tab_typ.round(1).to_string())

                towary  sprzedaz  udzial_sprzedazy_%
typ                                                 
nieregularny       639   50525.3                 4.3
regularny         2578  884922.3                75.1
sporadyczny       3553  144600.2                12.3
za_malo_danych    2832   10908.7                 0.9
zmienny            340   87715.6                 7.4


## Średnia kontra mediana w teście 2023 → 2024
#### Objaśnienie
**Po co.** Dla każdego towaru liczymy dwie stałe prognozy tygodniowe z 2023 i porównujemy je z tym, co faktycznie sprzedano w 2024.
**Co robi kod.**
- Bierzemy towary, które były w ofercie co najmniej 26 tygodni w 2023 (inaczej nie ma z czego uczyć).
- Średnia i mediana liczone są z tygodni **od pierwszej sprzedaży** (zera w trakcie życia towaru są prawdziwym popytem zero).
- Błąd to WAPE: suma |sprzedaż 2024 − prognoza| / suma sprzedaży 2024, osobno dla każdego typu. Dodajemy **obciążenie** (czy prognoza systematycznie zaniżona).
- `mediana=0_%`: jaka część towarów ma medianę zero, czyli prognozę „nie sprzeda się wcale”.
**Ważne ograniczenie WAPE.** Dla towarów sprzedających się rzadko WAPE nagradza niskie prognozy: zero w tygodniach bez sprzedaży daje mały błąd, mimo że prognoza „nie sprzeda się” jest bezużyteczna. Dlatego sam WAPE nie rozstrzyga. Gdy mediana wynosi 0 w połowie towarów typu lub więcej (`MEDIANA_ZERO_PROG`), wybieramy średnią niezależnie od WAPE.
**Jak czytać wynik.** Dla typów rzadkich mediana zwykle wypada zero (duży udział w kolumnie `mediana=0_%`), a jej obciążenie jest silnie ujemne. Dla regularnych obie metody są blisko siebie, a wygrywa ta o mniejszym WAPE. Kolumna `skosnosc` (mediana po towarach) mówi, czy rozkład jest wychylony w prawo: wartości powyżej 1 oznaczają silną skośność, czyli pojedyncze duże tygodnie ciągną średnią w górę.

In [11]:
MIN_TYG_UCZ = 26
A = tyg.values.astype(float)
n_akt = akt[:, :T].sum(axis=1)
ok = (n_akt >= MIN_TYG_UCZ) & (typ["typ"].values != "za_malo_danych")
Au = np.where(akt[:, :T], A[:, :T], np.nan)
with np.errstate(all="ignore"):
    f_sr = np.nanmean(Au[ok], axis=1)
    f_med = np.nanmedian(Au[ok], axis=1)
test = A[ok][:, T:]                                    # rzeczywista sprzedaż 2024
ty = typ.loc[ok, "typ"].values
sk = typ.loc[ok, "skosnosc"].values

wiersze = []
for t in ["regularny", "zmienny", "sporadyczny", "nieregularny"]:
    m = ty == t
    if m.sum() == 0:
        continue
    a = test[m]
    def wb(f):
        e = np.abs(a - f[m][:, None]).sum() / a.sum() * 100
        b = (f[m][:, None] * np.ones_like(a)).sum() / a.sum() * 100 - 100
        return e, b
    e_sr, b_sr = wb(f_sr); e_med, b_med = wb(f_med)
    wiersze.append({"typ": t, "towary": int(m.sum()), "mediana=0_%": 100 * (f_med[m] == 0).mean(), "skosnosc": np.nanmedian(sk[m]),
                    "WAPE_srednia_%": e_sr, "WAPE_mediana_%": e_med, "obciazenie_srednia_%": b_sr, "obciazenie_mediana_%": b_med})
wyn = pd.DataFrame(wiersze).set_index("typ")
wyn["wybor"] = np.where(wyn["WAPE_srednia_%"] <= wyn["WAPE_mediana_%"], "srednia", "mediana")
# Zasada: mediana równa 0 w połowie towarów typu oznacza prognozę "nie sprzeda się", która nic nie mówi o popycie.
# WAPE nagradza takie niskie prognozy (zero w tygodniach bez sprzedaży), więc samego WAPE nie wolno tu użyć do wyboru.
MEDIANA_ZERO_PROG = 50
wyn["wybor"] = np.where(wyn["mediana=0_%"] >= MEDIANA_ZERO_PROG, "srednia", wyn["wybor"])
def powod(r):
    if r["mediana=0_%"] >= MEDIANA_ZERO_PROG:
        return f"mediana daje zero w {r['mediana=0_%']:.0f} % towarów"
    if r["wybor"] == "mediana":
        return "rozkład skośny, mediana odporna na skoki" if r["skosnosc"] > 1 else "mniejszy błąd mediany"
    return "mniejszy błąd średniej"
wyn["uzasadnienie"] = wyn.apply(powod, axis=1)
print(wyn.round(1).to_string())

              towary  mediana=0_%  skosnosc  WAPE_srednia_%  WAPE_mediana_%  obciazenie_srednia_%  obciazenie_mediana_%    wybor                              uzasadnienie
typ                                                                                                                                                                       
regularny       2464          0.0       0.7            49.3            48.2                   7.9                   1.8  mediana                     mniejszy błąd mediany
zmienny          321          0.0       1.7            80.7            74.0                  12.2                 -17.1  mediana  rozkład skośny, mediana odporna na skoki
sporadyczny     3238         67.2       1.8           156.8           112.5                  56.6                 -10.1  srednia          mediana daje zero w 67 % towarów
nieregularny     604         57.8       2.9           183.7           109.4                  78.6                 -38.4  srednia          mediana

## Wniosek i zapis
#### Objaśnienie
**Po co.** Zapisujemy decyzję dla każdego typu, żeby kolejne kroki (okno N, typy popytu) mogły z niej korzystać.
**Jak czytać wynik.** Reguła jest jedna na typ: jeśli średnia i mediana są blisko siebie (różnica poniżej 1 punktu procentowego), wybór jest obojętny i przyjmujemy średnią, bo zachowuje sumę sprzedaży. Wynik testu dotyczy jednego przejścia 2023 → 2024, więc różnice rzędu kilku punktów traktuj ostrożnie.

In [12]:
wyn["roznica_pp"] = (wyn["WAPE_srednia_%"] - wyn["WAPE_mediana_%"]).abs()
wyn.loc[wyn["roznica_pp"] < 1, "wybor"] = "srednia"                  # remis: średnia zachowuje sumę sprzedaży
print(wyn[["wybor", "uzasadnienie", "roznica_pp"]].round(1).to_string())
ew.zapisz_wynik(wyn, "wynik_02_srednia_mediana.csv")

                wybor                              uzasadnienie  roznica_pp
typ                                                                        
regularny     mediana                     mniejszy błąd mediany         1.1
zmienny       mediana  rozkład skośny, mediana odporna na skoki         6.8
sporadyczny   srednia          mediana daje zero w 67 % towarów        44.3
nieregularny  srednia          mediana daje zero w 58 % towarów        74.2
Zapisano: dane/eda/wynik_02_srednia_mediana.csv (4 wierszy)


'dane/eda/wynik_02_srednia_mediana.csv'